**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sparse Coding & Dictionary Learning

[Compressed Sensing](./Compressed_Sensing.ipynb) assumed a known sparsifying basis. This sequel asks two harder questions: how do you find the sparse code *greedily and fast* (OMP), and — the big one — can you **learn the dictionary itself from data** (K-SVD)? Both verified on planted-truth problems where we know the answer.

## 1. Pre-requisites

[Compressed Sensing](./Compressed_Sensing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2/S4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Greedy Pursuit: OMP* (~40 min)
**Goal:** build the sparse code one atom at a time; verify exact recovery of a planted support.
**Builds on:** [Compressed Sensing](./Compressed_Sensing.ipynb). &nbsp; **Feeds into:** Session 2 (dictionary learning).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Greedy Pursuit — OMP</b></summary>

**Timing (~40 min).** 8 min greedy versus convex · 10 min the algorithm, especially the "orthogonal" · 10 min the exact-recovery oracle · 12 min the cliff.

**Position OMP against [Compressed Sensing](./Compressed_Sensing.ipynb)'s L1.** Same problem, opposite philosophy. L1 is principled — a convex relaxation with recovery theorems — but needs an iterative solver. OMP is the engineer's answer: pick the atom most correlated with the residual, re-fit, repeat $K$ times. It is transparent, fast, and gives you the support explicitly rather than as a by-product of thresholding. Ask which you would rather debug at 2 a.m.

**Make sure the "orthogonal" earns its name.** Plain matching pursuit picks an atom and subtracts its contribution; OMP re-solves least squares over *all* chosen atoms at every step. The consequence is that the residual stays perpendicular to everything already selected, so no atom is ever chosen twice and each iteration makes strict progress. Students skim this as an implementation detail — it is the difference between an algorithm that converges in $K$ steps and one that dithers. Point at `np.linalg.lstsq(Ds, y)` inside the loop.

**Set up the oracle as a bet.** 64 measurements, 256 atoms, 6 nonzeros: the system is massively underdetermined, and there are $\binom{256}{6} \approx 3\times10^{11}$ possible supports. Ask the room whether a greedy algorithm — which never reconsiders a choice — can find the right one. It does, exactly, and the coefficient error is 2.4e-15. Greedy algorithms usually come with an approximation-ratio caveat; here, under incoherence, greed is provably optimal.

**The cliff is the real content of the session — protect its time.** Recovery rate against sparsity is not a gentle slope; it is near-1 and then it collapses. Ask the room to predict the shape before you plot it. Phase transitions like this are characteristic of sparse recovery, and the mechanism is worth naming: with a random 64×256 dictionary, coherence between atoms sets a threshold, and beyond it a wrong atom can correlate with the residual better than a right one. One wrong pick early is unrecoverable, because OMP never backtracks — which is exactly the price of greed.

**Ask the room.** "Where does the cliff have to be, roughly?" It cannot exceed $K = 64$, since 64 measurements cannot determine more than 64 unknowns even with a perfect method. The observed collapse arrives well before that, and the gap between the information-theoretic limit and the algorithm's limit is where L1 methods and the recovery theorems live.

**If the demo is slow.** The cliff sweep is 150 trials at each of 7 sparsity levels, each running OMP — expect a few seconds. Lowering the trial count makes the curve noisier without changing where the cliff sits.
</details>

## 2. One Atom at a Time

💡 **Intuition.** L1 minimization is principled but iterative-solver-shaped. **Orthogonal Matching Pursuit** is the greedy engineer's answer: repeatedly pick the atom most correlated with the residual, then re-fit *all* chosen atoms by least squares (the 'orthogonal' — the residual stays perpendicular to everything chosen, so no atom is picked twice). $K$ iterations, each a correlation + a small solve; exact recovery when the dictionary is incoherent enough.

In [2]:
def omp(D, y, K):
    resid = y.copy(); support = []
    for _ in range(K):
        support.append(int(np.argmax(np.abs(D.T @ resid))))
        Ds = D[:, support]
        coef, *_ = np.linalg.lstsq(Ds, y, rcond=None)
        resid = y - Ds @ coef
    x = np.zeros(D.shape[1]); x[support] = coef
    return x, sorted(support)

# ORACLE: planted 6-sparse code in a random 64×256 dictionary — recover it exactly
n_dim, n_atoms, K = 64, 256, 6
D = rng.standard_normal((n_dim, n_atoms)); D /= np.linalg.norm(D, axis=0)
true_supp = sorted(rng.choice(n_atoms, K, replace=False).tolist())
x_true = np.zeros(n_atoms); x_true[true_supp] = rng.standard_normal(K) * 2
y = D @ x_true

x_hat, supp = omp(D, y, K)
print(f"true support:      {true_supp}")
print(f"OMP support:       {supp}")
print(f"coefficient error: {np.abs(x_hat - x_true).max():.2e}")
assert supp == true_supp

true support:      [11, 33, 150, 214, 234, 238]
OMP support:       [11, 33, 150, 214, 234, 238]
coefficient error: 2.44e-15


**What just happened.** OMP recovered the support **exactly** — `[11, 33, 150, 214, 234, 238]` in both rows — with a coefficient error of **2.4e-15**, which is machine precision. The `assert` guarantees it.

Take the measure of that. The dictionary is 64 × 256, so the system is underdetermined by a factor of four, and there are $\binom{256}{6} \approx 3\times 10^{11}$ possible 6-element supports. A greedy algorithm — one that picks an atom, never reconsiders, and repeats six times — found the right one out of three hundred billion. Greedy methods normally come with an approximation-ratio caveat attached; here, under sufficient incoherence, greed is provably exact.

**Why the coefficient error is 1e-15 rather than 1e-6.** Once the *support* is correct, the remaining problem is an ordinary overdetermined least-squares fit of 6 unknowns to 64 equations, which `lstsq` solves to machine precision. So the two printed results are really one discrete claim and one trivial consequence: get the support right and the coefficients are free. This is why support recovery, not coefficient accuracy, is the meaningful measure in sparse problems.

**The "orthogonal" is doing real work.** Plain matching pursuit picks an atom and subtracts its contribution. OMP re-solves least squares over *all* selected atoms at every iteration — the `np.linalg.lstsq(Ds, y)` inside the loop. That keeps the residual perpendicular to everything already chosen, so no atom is ever selected twice and every iteration strictly reduces the residual. Without it the algorithm can revisit atoms and dither; with it, $K$ iterations suffice.

**And note what it never does: backtrack.** Each atom is chosen by a single correlation test against the current residual, and that choice is permanent. When the dictionary is incoherent enough, the correct atom always wins that test and the greed is harmless. When it is not, one early mistake is unrecoverable — the algorithm keeps building on a wrong foundation and the support comes out wrong. That fragility is not visible in this clean run, and it is exactly what the next cell measures.

In [3]:
# and its breaking point: recovery probability vs sparsity level (the coherence wall)
def trial(K_s):
    ts = sorted(rng.choice(n_atoms, K_s, replace=False).tolist())
    x = np.zeros(n_atoms); x[ts] = rng.standard_normal(K_s)
    _, s = omp(D, D @ x, K_s)
    return s == ts
Ks = [4, 8, 12, 16, 20, 24, 28]
rates = [np.mean([trial(K_s) for _ in range(150)]) for K_s in Ks]
plt.figure(figsize=(7, 2.6))
plt.plot(Ks, rates, "o-")
plt.xlabel("sparsity K"); plt.ylabel("exact recovery rate")
plt.title(f"64 measurements, 256 atoms: greedy recovery falls off a cliff")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2983624/2767358051.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


**What just happened.** Recovery rate against sparsity, and the shape is the result: near-certain success at low $K$, then a **collapse** over a narrow range. It is not a gentle decline in accuracy — it is a phase transition between "works essentially always" and "fails essentially always."

**Why a cliff rather than a slope.** Success here is a *discrete* event: either every one of the $K$ selected atoms is correct or the support is wrong. OMP picks each atom by a single correlation test and never backtracks, so one early mistake is unrecoverable — the algorithm keeps re-fitting on a wrong foundation. As $K$ grows, two things worsen together: more chances to make that first mistake, and a residual increasingly likely to correlate better with some wrong atom than with a remaining right one, because a random 64×256 dictionary has non-zero coherence between atoms. Once the failure probability per pick crosses a threshold, compounding over $K$ picks does the rest.

**Locate the two limits.** There is a hard information-theoretic ceiling at $K = 64$: 64 measurements cannot determine more than 64 unknowns, whatever algorithm you use. The observed cliff arrives well below that. The gap between them is the price of greed — L1 minimisation from [Compressed Sensing](./Compressed_Sensing.ipynb) generally pushes further, at the cost of an iterative solve, and the recovery theorems in that literature are about exactly this gap.

**Phase transitions are the normal behaviour in this field, not a quirk of OMP.** Compressed sensing, community detection, and matrix completion all show them, and the practical consequence matters: you cannot extrapolate from a working regime. A system tested at $K = 6$ tells you nothing about $K = 20$, because the degradation is not gradual. If you are deploying sparse recovery, find your cliff empirically and stay well clear of it — the region just below it is where performance is technically fine and one unlucky dataset ruins you.

Note also what is held fixed: the dictionary is 64 × 256 throughout, so this curve is specific to that shape and to random Gaussian atoms. A more incoherent dictionary moves the cliff right; a larger, more redundant one moves it left.

---
### 🕐 Session 2 of 3 — *K-SVD: Learning the Dictionary* (~40 min)
**Goal:** alternate sparse coding and per-atom SVD updates; recover a PLANTED dictionary.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (denoising with learned atoms).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: K-SVD — Learning the Dictionary</b></summary>

**Timing (~40 min).** 8 min the reframe · 12 min the alternating algorithm · 10 min the SVD update specifically · 10 min the oracle and its caveats.

**Open with the reframe, because it is the point of the workshop.** Every sparse method so far assumed a dictionary was *given* — Fourier, DCT, wavelets, all designed by mathematicians on the grounds that natural signals ought to be sparse in them. K-SVD asks the obvious next question: why guess, when you have data? Representation stops being a design choice and becomes a fit. That framing also explains why this workshop sits just before [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) in the curriculum — it is the same idea, pre-neural.

**The algorithm is alternating minimisation; name that pattern.** Fix the dictionary, solve for codes (OMP, Session 1). Fix the codes, solve for the dictionary. Repeat. Students have met this shape before in k-means and EM, and saying so is worth thirty seconds — as is the consequence: alternating minimisation converges to a *local* optimum, and initialisation matters. `D` starts from randomly chosen training signals for exactly that reason.

**The per-atom SVD update is the clever part — go slowly.** To update atom $j$: restrict to the signals that actually use it, form the residual matrix $E$ with atom $j$'s own contribution *added back in*, and take the rank-one SVD. Ask why rank-one — because [Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) says the leading singular pair is the best rank-one approximation, so this is provably the best single direction to explain what is left. And note it updates the atom *and* its coefficients simultaneously, which is why K-SVD converges faster than gradient methods on the same objective.

**Point at the dead-atom branch.** `if len(users) == 0` re-randomises an atom nobody uses. That is not defensive padding — atom death is a real failure mode of K-SVD, and without the fix the effective dictionary silently shrinks. Worth naming because students otherwise read it as an edge case.

**Set up the oracle as the audit most demos skip.** Plenty of dictionary-learning demos show pretty atoms and stop. Here we *plant* a dictionary, generate data from it, and then count how many atoms are recovered to $|\cos| > 0.98$ — matching up to sign and permutation, since neither is identifiable. 20 of 20 is a strong claim, not a vibe.

**Be honest that the conditions are generous, and say what breaks.** 3000 training signals for 20 atoms in 16 dimensions, 45 iterations, and noise at 0.01. Starve any of those and K-SVD drops atoms into local minima — the printed line says so, and the `assert` is deliberately set at 16 rather than 20 because the result is not guaranteed. Running with `n_iter=5` or 300 signals is a good live demonstration that the 20/20 is earned by the budget, not automatic.
</details>

## 3. Where Do Atoms Come From?

💡 **Intuition.** Wavelets are a guess; data can vote. **K-SVD** alternates: (1) sparse-code every training signal with the current dictionary (OMP), (2) update each atom — restrict to the signals that *use* it, and set the atom (and its coefficients) to the **rank-one SVD** of their residual matrix — the best single direction explaining what's left ([Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) again). The audit most demos skip: plant a dictionary, generate data from it, and count how many atoms K-SVD *actually recovers*.

In [4]:
def ksvd(Y, n_atoms, K_s, n_iter=30):
    D = Y[:, rng.choice(Y.shape[1], n_atoms, replace=False)].astype(float)
    D /= np.linalg.norm(D, axis=0)
    for it in range(n_iter):
        X = np.stack([omp(D, y, K_s)[0] for y in Y.T], axis=1)      # sparse coding
        for j in range(n_atoms):                                     # atom-by-atom update
            users = np.nonzero(X[j])[0]
            if len(users) == 0:
                D[:, j] = rng.standard_normal(D.shape[0]); D[:, j] /= np.linalg.norm(D[:, j]); continue
            E = Y[:, users] - D @ X[:, users] + np.outer(D[:, j], X[j, users])
            U, s, Vt = np.linalg.svd(E, full_matrices=False)
            D[:, j] = U[:, 0]; X[j, users] = s[0] * Vt[0]
    return D, X

# ORACLE: plant a 20-atom dictionary in R^16, generate 3-sparse data, recover the atoms
n_dim2, n_at2, K2 = 16, 20, 3
D_true = rng.standard_normal((n_dim2, n_at2)); D_true /= np.linalg.norm(D_true, axis=0)
Y = np.stack([D_true[:, rng.choice(n_at2, K2, replace=False)] @ rng.standard_normal(K2)
              for _ in range(3000)], axis=1)
Y += 0.01 * rng.standard_normal(Y.shape)

D_learn, _ = ksvd(Y, n_at2, K2, n_iter=45)
# match learned atoms to true atoms (up to sign and permutation)
sims = np.abs(D_learn.T @ D_true)
recovered = int((sims.max(0) > 0.98).sum())
print(f"planted atoms recovered (|cos| > 0.98): {recovered} / {n_at2}")
print("(with ample data + iterations: all 20; starve either and K-SVD drops atoms to local minima)")
assert recovered >= 16

planted atoms recovered (|cos| > 0.98): 20 / 20
(with ample data + iterations: all 20; starve either and K-SVD drops atoms to local minima)


**What just happened.** **20 of 20** planted atoms recovered to $|\cos| > 0.98$. Not "the learned dictionary looks reasonable" — every single direction of the true dictionary was found, matched up to sign and permutation, neither of which is identifiable.

This is the audit most dictionary-learning demos skip. The usual presentation trains on real images, displays a grid of Gabor-like atoms, and invites you to find them plausible. Plausibility is not verification. Here a dictionary was *planted*, data was generated from it, and the question "did we get the atoms back?" has a checkable answer.

**How the alternating scheme gets there.** Fix $D$, sparse-code every signal with OMP; fix the codes, update each atom. That is alternating minimisation, the same shape as k-means and EM — with the same consequence, which is convergence to a *local* optimum that depends on initialisation. `D` starts from randomly chosen training signals precisely because that is a better starting point than noise.

**The per-atom update is the elegant part.** For atom $j$, restrict to the signals that actually use it, form the residual with atom $j$'s own contribution added back in, and take the **rank-one SVD**. By [Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), the leading singular pair is the provably best rank-one approximation of that residual — so the update is not a heuristic but the optimal single direction to explain what remains. It also updates the atom and its coefficients *simultaneously*, which is why K-SVD converges in tens of iterations where gradient descent on the same objective takes far longer.

**One line worth noticing.** `if len(users) == 0` re-randomises any atom that no signal uses. Atom death is a genuine K-SVD failure mode, not a hypothetical edge case: without that branch the effective dictionary silently shrinks and you learn fewer atoms than you asked for, with nothing in the output to tell you.

**And the conditions here are generous — the result is earned, not automatic.** 3000 training signals for 20 atoms in 16 dimensions, 45 iterations, and noise at 0.01. The printed note says what happens if you starve any of that, and the `assert` is deliberately set at 16 rather than 20 because a perfect score is not guaranteed. Try `n_iter=5`, or 300 training signals instead of 3000: recovery drops, because alternating minimisation lands in a local optimum with some true atoms never found and some learned atoms sitting between two real ones. That is the honest behaviour of the method, and knowing the failure shape is more useful than the clean number.

---
### 🕐 Session 3 of 3 — *Denoising with Learned Atoms* (~35 min)
**Goal:** the payoff: a dictionary learned from noisy patches beats a fixed basis at denoising.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Denoising with Learned Atoms</b></summary>

**Timing (~35 min).** 10 min why sparse approximation denoises · 10 min the comparison setup · 10 min the result, including the negative number · 5 min the lineage.

**Board first — the one-sentence mechanism.** Code each noisy patch with only 3 atoms and rebuild. Whatever the dictionary can express in 3 atoms survives; noise, which is sparse in *no* dictionary, cannot be captured and dies. Denoising here is a side effect of being forced to be brief. That framing generalises — it is the same reason low-rank approximation denoises matrices.

**Set up the comparison honestly before running.** Same patches, same noise, same sparsity budget of 3 atoms, same OMP solver. The *only* difference is which dictionary. Say that explicitly; it makes the result a controlled experiment rather than a demonstration.

**The DCT's negative result is the best teaching moment here — do not skip past it.** DCT gives **−0.8 dB**: the "denoised" output is *worse* than the noisy input. Ask the room how that is possible. The answer is that the error has two parts — noise removed and signal destroyed — and with only 3 DCT atoms the patches (edges, narrow Gaussian bumps) simply cannot be represented. A step edge needs many DCT coefficients, so a 3-term DCT approximation throws away real signal faster than it removes noise. The learned dictionary has atoms *shaped like* edges and bumps, so 3 of them suffice.

**Then the generalisable lesson.** Sparse approximation only denoises if your signal really is sparse in that dictionary. Applied in the wrong basis it is just lossy compression, and it can be worse than doing nothing. Students tend to absorb "sparsity denoises" as unconditional; the −0.8 dB is the counterexample that fixes that.

**Ask the room.** "The DCT is a complete orthonormal basis for $\mathbb{R}^{16}$ — it can represent these patches *exactly*. So why does it lose?" Because completeness is not sparsity. Exact representation may need all 16 coefficients; the question is whether 3 suffice. Distinguishing "can represent" from "can represent *briefly*" is the conceptual core of the whole workshop.

**Note the fair-play detail.** `D_data` is trained on *noisy* patches (`train + 0.02*randn`) from the same family, never on the test patches, and the DCT is the full 16-atom basis. Nobody is being handicapped. Worth saying, since a sceptical student should be asking.

**Close on lineage.** K-SVD denoising was state of the art for roughly a decade and is the conceptual ancestor of every learned-representation denoiser since — including the score networks in [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb), which are denoisers whose "dictionary" is a neural network. The idea that representation should be *learned* rather than designed starts here.
</details>

## 4. The Payoff

💡 **Intuition.** Denoise by *sparse approximation*: code each noisy patch with a few atoms, rebuild — whatever the dictionary can express survives, and noise (which is sparse in **no** dictionary) dies. Learned atoms fit the data's actual structure better than any fixed basis fits it, so at equal sparsity they keep more signal. This pipeline (K-SVD denoising) was state-of-the-art for a decade and is the conceptual ancestor of every learned-representation denoiser since — including [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb).

In [5]:
# 1-D 'patches' from a piecewise-smooth signal family; compare DCT vs learned dictionary
from scipy.fft import dct
def make_patch():
    t = np.linspace(0, 1, n_dim2)
    kind = rng.integers(3)
    if kind == 0:  return np.sin(2*np.pi*rng.uniform(1, 3)*t + rng.uniform(0, 6))
    if kind == 1:  return np.sign(t - rng.uniform(0.2, 0.8)) * rng.uniform(0.5, 1)
    return np.exp(-((t - rng.uniform(0.2, 0.8))/0.08)**2)

train = np.stack([make_patch() for _ in range(2000)], axis=1)
D_data, _ = ksvd(train + 0.02*rng.standard_normal(train.shape), 24, 3, n_iter=20)
D_dct = dct(np.eye(n_dim2), norm="ortho", axis=0)          # full 16-atom DCT basis

# denoise fresh noisy patches by 3-sparse OMP approximation in each dictionary
sigma_n = 0.25
test = np.stack([make_patch() for _ in range(300)], axis=1)
noisy = test + sigma_n * rng.standard_normal(test.shape)
def denoise(D_use):
    out = np.stack([D_use @ omp(D_use, y, 3)[0] for y in noisy.T], axis=1)
    return 10*np.log10(np.var(noisy - test) / np.var(out - test))
print(f"input SNR {10*np.log10(np.var(test)/sigma_n**2):.1f} dB")
print(f"denoising gain — DCT basis: {denoise(D_dct):+.1f} dB   learned dictionary: {denoise(D_data):+.1f} dB")

input SNR 7.8 dB
denoising gain — DCT basis: -0.8 dB   learned dictionary: +2.5 dB


**What just happened.** From a 7.8 dB input, the learned dictionary gains **+2.5 dB** while the DCT basis gives **−0.8 dB**. The DCT result is not a small win — it is *negative*: the "denoised" output is worse than the noisy input it started from.

This is a properly controlled experiment. Same patches, same noise realisation, same sparsity budget of 3 atoms, same OMP solver. The only variable is which dictionary, so the 3.3 dB spread is attributable to representation alone.

**Why sparse approximation denoises at all.** Force each patch to be built from just 3 atoms. Structure the dictionary can express in 3 atoms survives; noise is sparse in *no* dictionary, so it cannot be captured and is discarded. Denoising is a side effect of being made to be brief.

**And why the DCT loses so badly that it goes negative.** The error has two parts: noise removed (good) and signal destroyed (bad). Our patches are sinusoids, step edges, and narrow Gaussian bumps. A step edge is famously *not* sparse in the DCT — it needs many coefficients to build a discontinuity, which is Gibbs' phenomenon from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) wearing a different hat. Restricted to 3 DCT atoms, the approximation destroys more real signal than it removes noise, and the balance comes out negative. The learned dictionary contains atoms *shaped like* edges and bumps, because it was fitted to patches of exactly that kind, so 3 of them are plenty.

**The lesson generalises past this cell.** "Sparsity denoises" is not unconditional — it holds only if the signal really is sparse in the dictionary you chose. In the wrong basis, sparse approximation is simply lossy compression and can be worse than doing nothing at all. The −0.8 dB is the counterexample worth remembering.

Worth pushing on the natural objection: the DCT is a *complete orthonormal basis* for $\mathbb{R}^{16}$ and can represent these patches exactly. So why does it lose? Because completeness is not sparsity. Exact representation may require all 16 coefficients; the question is whether **3** suffice. Distinguishing "can represent" from "can represent *briefly*" is the conceptual core of the entire workshop.

**And the comparison is fair.** `D_data` was trained on *noisy* patches from the same family and never saw the test set; the DCT is its full 16-atom basis, not a truncation. Neither side is handicapped — the learned dictionary wins because it knows what these signals look like.

**Lineage.** This pipeline was state of the art for about a decade, and it is the direct ancestor of every learned-representation denoiser since. The score networks in [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb) are denoisers whose dictionary is a neural network rather than a matrix — same principle, learned representation, vastly more capacity.

## 5. Conclusion

OMP builds codes greedily and provably recovers planted supports; K-SVD recovers planted *dictionaries* atom-for-atom; and sparse approximation in the learned dictionary is a denoiser that knows your data. Representation is no longer a design choice — it's a fit.

---
## Where next

- [Compressed Sensing](./Compressed_Sensing.ipynb) — the convex counterpart.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants of this exact idea.